In [156]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupShuffleSplit
from scipy.sparse import hstack
from sklearn.metrics import classification_report
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report,
)
import plotly.graph_objects as go
import sys
sys.path.append('..')
from funs import *

In [157]:
data = get_df()
data

,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target,Nationality,Dropout
0,Single,2nd phase - general contingent,5,Animation and Multimedia Design,Daytime,Secondary education,122.0,Basic Education 3rd Cycle (9th/10th/11th Year)...,Other - 11th Year of Schooling,"Personal Services, Security and Safety Workers...",...,0,0,0.000000,0,10.8,1.4,1.74,Dropout,Portugese,True
1,Single,International student (bachelor),1,Tourism,Daytime,Secondary education,160.0,Secondary Education - 12th Year of Schooling o...,Higher Education - Degree,Intermediate Level Technicians and Professions,...,6,6,13.666667,0,13.9,-0.3,0.79,Graduate,Portugese,False
2,Single,1st Phase General Contingent,5,Communication Design,Daytime,Secondary education,122.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,Unskilled Workers,...,0,0,0.000000,0,10.8,1.4,1.74,Dropout,Portugese,True
3,Single,2nd phase - general contingent,2,Journalism and Communication,Daytime,Secondary education,122.0,Basic Education 2nd Cycle (6th/7th/8th Year) o...,Basic education 1st cycle (4th/5th year) or eq...,"Personal Services, Security and Safety Workers...",...,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate,Portugese,False
4,Married,Over 23 years old,1,Social Service (evening attendance),Evening,Secondary education,100.0,Basic education 1st cycle (4th/5th year) or eq...,Basic Education 2nd Cycle (6th/7th/8th Year) o...,Unskilled Workers,...,6,6,13.000000,0,13.9,-0.3,0.79,Graduate,Portugese,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,Single,1st Phase General Contingent,6,Journalism and Communication,Daytime,Secondary education,125.0,Secondary Education - 12th Year of Schooling o...,Secondary Education - 12th Year of Schooling o...,"Personal Services, Security and Safety Workers...",...,8,5,12.666667,0,15.5,2.8,-4.06,Graduate,Portugese,False
4420,Single,1st Phase General Contingent,2,Journalism and Communication,Daytime,Secondary education,120.0,Secondary Education - 12th Year of Schooling o...,Secondary Education - 12th Year of Schooling o...,Unskilled Workers,...,6,2,11.000000,0,11.1,0.6,2.02,Dropout,Russian,True
4421,Single,1st Phase General Contingent,1,Nursing,Daytime,Secondary education,154.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,Unskilled Workers,...,9,1,13.500000,0,13.9,-0.3,0.79,Dropout,Portugese,True
4422,Single,1st Phase General Contingent,1,Management,Daytime,Secondary education,180.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,"Skilled Workers in Industry, Construction and ...",...,6,5,12.000000,0,9.4,-0.8,-3.12,Graduate,Portugese,False


In [158]:
# groupings
data['Single?'] = data['Marital Status'] == 'Single'

father_occ_groups = {
    'blank': ['(blank)', 'Other Situation', 'Unskilled Workers', 'fathers_qualification'],
    'management': ['Adminstrative staff', 
                   'Directors of administrative and commercial services',
                   'Other administrative support staff'],
    'military': ['Armed Forces Officers', 
                 'Armed Forces Profcessions', 
                 'Armed Forces Sergeants',
                 'Other Armed Forces personnel'],
    'finance': ['Data, accounting, statistical, financial services and registry-related operators',
                'Office workers, secretaries in general and data processing operators',
                'Specialists in finance, accounting, administrative organization, public and commercial relations'],
    'agriculture': ['Farmers and Skilled Workers in Agriculture, Fisheries and Forestry', 
                    'Farmers, livestock keepers, fishermen, hunters and gatherers, subsistence', 
                    'Fixed plant and machine operators',
                    'Market-oriented farmers and skilled agricultural and animal production workers',
                    'Unskilled workers in agriculture, animal production, fisheries and forestry'],
    'service': ['Hotel, catering, trade and other services directors',
                'Intermediate level technicians from legal, social, sports, cultural and similar services',
                'Meal preparation assistants',
                'Personal Services, Security and Safety Workers and Sellers',
                'Personal care workers and the like',
                'Protection and security services personnel',
                'Street vendors (except food) and street service providers',
                'personal service workers',
                'sellers'],
    'technology': ['Information and communication technology technicians',
                   'Intermediate Level Technicians and Professions',
                   'Intermediate level science and engineering technicians and professions',
                   'Technicians and professionals, of intermediate level of health'],
    'line': ['Installation and Machine Operators and Assembly Workers',
             'Vehicle drivers and mobile equipment operators',
             'Workers in food processing, woodworking, clothing and other industries and crafts',
             'assembly workers'],
    'legal': ['Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers'],
    'blue_collar': ['Skilled Workers in Industry, Construction and Craftsmen',
                    'Skilled construction workers and the like, except electricians',
                    'Skilled workers in electricity and electronics',
                    'Skilled workers in metallurgy, metalworking and similar',
                    'Unskilled workers in extractive industry, construction, manufacturing and transport'],
    'academic': ['Specialists in Intellectual and Scientific Activities',
                 'Specialists in the physical sciences, mathematics, engineering and related techniques',
                 'Student',
                 'teachers'],
                 
}

edu = {
    'none': ["Can't read or write", 'Unknown'],
    'elementary': ['Basic education 1st cycle (4th/5th year) or equiv.',
                   'Can read without having a 4th year of schooling'],
    'middle': ['7th Year (Old)',
               '7th year of schooling',
               '8th year of schooling',
               'Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv.'],
    'high': ['10th Year of Schooling',
             '11th Year of Schooling - Not Completed',
             '12th Year of Schooling - Not Completed',
             '2nd cycle of the general high school course',
             '9th Year of Schooling - Not Completed',
             'Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv.',
             'Other - 11th Year of Schoolin',
             'Secondary Education - 12th Year of Schooling or Eq.'],
    'college': ['Frequency of Higher Education',
                "Higher Education - Bachelor's Degree",
                'Higher Education - Degree',
                'Higher Education - Doctorate',
                'Higher Education - Doctorate (3rd cycle)',
                'Higher Education - Master (2nd cycle)',
                "Higher Education - Master's",
                "Higher education - degree (1st cycl"],
    'other': ['General commerce course',
              'Professional higher technical course',
              'Specialized higher studies course',
              'Technical-professional course',
              'Technological specialization course']
}


data['Father Occ Group'] = data["Father's occupation"].map(father_occ_groups).fillna('other')
data['Mother Occ Group'] = data["Mother's occupation"].map(father_occ_groups).fillna('other')
data['Father Edu Group'] = data["Father's qualification"].map(edu).fillna('none')
data['Mother Edu Group'] = data["Mother's qualification"].map(edu).fillna('none')
data

,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Mother's qualification,Father's qualification,Mother's occupation,...,Inflation rate,GDP,Target,Nationality,Dropout,Single?,Father Occ Group,Mother Occ Group,Father Edu Group,Mother Edu Group
0,Single,2nd phase - general contingent,5,Animation and Multimedia Design,Daytime,Secondary education,122.0,Basic Education 3rd Cycle (9th/10th/11th Year)...,Other - 11th Year of Schooling,"Personal Services, Security and Safety Workers...",...,1.4,1.74,Dropout,Portugese,True,True,other,other,none,none
1,Single,International student (bachelor),1,Tourism,Daytime,Secondary education,160.0,Secondary Education - 12th Year of Schooling o...,Higher Education - Degree,Intermediate Level Technicians and Professions,...,-0.3,0.79,Graduate,Portugese,False,True,other,other,none,none
2,Single,1st Phase General Contingent,5,Communication Design,Daytime,Secondary education,122.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,Unskilled Workers,...,1.4,1.74,Dropout,Portugese,True,True,other,other,none,none
3,Single,2nd phase - general contingent,2,Journalism and Communication,Daytime,Secondary education,122.0,Basic Education 2nd Cycle (6th/7th/8th Year) o...,Basic education 1st cycle (4th/5th year) or eq...,"Personal Services, Security and Safety Workers...",...,-0.8,-3.12,Graduate,Portugese,False,True,other,other,none,none
4,Married,Over 23 years old,1,Social Service (evening attendance),Evening,Secondary education,100.0,Basic education 1st cycle (4th/5th year) or eq...,Basic Education 2nd Cycle (6th/7th/8th Year) o...,Unskilled Workers,...,-0.3,0.79,Graduate,Portugese,False,False,other,other,none,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,Single,1st Phase General Contingent,6,Journalism and Communication,Daytime,Secondary education,125.0,Secondary Education - 12th Year of Schooling o...,Secondary Education - 12th Year of Schooling o...,"Personal Services, Security and Safety Workers...",...,2.8,-4.06,Graduate,Portugese,False,True,other,other,none,none
4420,Single,1st Phase General Contingent,2,Journalism and Communication,Daytime,Secondary education,120.0,Secondary Education - 12th Year of Schooling o...,Secondary Education - 12th Year of Schooling o...,Unskilled Workers,...,0.6,2.02,Dropout,Russian,True,True,other,other,none,none
4421,Single,1st Phase General Contingent,1,Nursing,Daytime,Secondary education,154.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,Unskilled Workers,...,-0.3,0.79,Dropout,Portugese,True,True,other,other,none,none
4422,Single,1st Phase General Contingent,1,Management,Daytime,Secondary education,180.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,"Skilled Workers in Industry, Construction and ...",...,-0.8,-3.12,Graduate,Portugese,False,True,other,other,none,none


In [ ]:
categorical = [
    'Daytime/evening attendance', 
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder',
    # 'Application mode',
    # 'Single?'
    # 'Father Occ Group',
    # 'Mother Occ Group'
    # 'Father Edu Group',
    # 'Mother Edu Group'
]
numerical = ['Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)']
target = ['Dropout']

In [160]:
TEST_SIZE = 0.2

X = data[categorical + numerical]
y = (data['Dropout']).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y
)

In [161]:
N_NEIGHBORS = 15
RANDOM_STATE = 42

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('num', StandardScaler(), numerical)
    ]
)

model = KNeighborsClassifier(
    n_neighbors=N_NEIGHBORS,
    n_jobs=-1
)

pipe = Pipeline([
    ("preprocessor", preprocessor), 
    ("knn", model)
])

pipe.fit(X_train, y_train)

predictions = pipe.predict(X_test)
probabilities = pipe.predict_proba(X_test)

y_preds = pipe.predict(X_test)
print(f"KNN with {N_NEIGHBORS} neighbors")
print(classification_report(y_test, y_preds))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_preds))

KNN with 15 neighbors
              precision    recall  f1-score   support

           0       0.86      0.95      0.90       601
           1       0.87      0.67      0.75       284

    accuracy                           0.86       885
   macro avg       0.86      0.81      0.83       885
weighted avg       0.86      0.86      0.86       885

Confusion Matrix:
[[573  28]
 [ 95 189]]


In [162]:
def plot_roc_curves(y_true, scores_dict, out_html="roc_curve.html"):
    fig = go.Figure()
    for name, y_score in scores_dict.items():
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{name} (AUC={auc:.3f})"))
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                             name="Chance", line=dict(dash="dash")))
    fig.update_layout(
        title="ROC Curve Comparison",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        template="plotly_white",
        legend_title_text=None,
        width=900, height=550
    )
    fig.write_html(out_html, include_plotlyjs="cdn")

In [163]:
def plot_pr_curves(y_true, scores_dict, out_html="pr_curve.html"):
    fig = go.Figure()
    pos_rate = (y_true.sum() / len(y_true)) if len(y_true) else 0.0
    for name, y_score in scores_dict.items():
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines",
                                 name=f"{name} (AP={ap:.3f})"))
    fig.add_trace(go.Scatter(x=[0,1], y=[pos_rate, pos_rate], mode="lines",
                             name=f"Baseline (pos rate={float(pos_rate):.3f})",
                             line=dict(dash="dash")))
    fig.update_layout(
        title="Precision–Recall Curve",
        xaxis_title="Recall",
        yaxis_title="Precision",
        template="plotly_white",
        legend_title_text=None,
        width=900, height=550
    )
    fig.write_html(out_html, include_plotlyjs="cdn")

In [164]:
plot_roc_curves(y_test, {"KNN": probabilities[:, 1]}, out_html="knn2_roc_curve.html")
plot_pr_curves(y_test, {"KNN": probabilities[:, 1]}, out_html="knn2_pr_curve.html")